# 3회차 — Qwen 모델 추론, 프롬프트, 생성 파라미터

**파이썬 심화 | CPU 실습**

- 모델: `Qwen/Qwen2.5-0.5B-Instruct`
- 환경: VS Code + Jupyter Notebook + `.venv`


## 1. 실습 환경 확인

In [1]:
# 현재 Python 및 설치된 라이브러리 확인
import sys
import time
import torch
import transformers
import pandas as pd

print("Python 경로:", sys.executable)
print("Python 버전:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())


Python 경로: c:\python-llm-advanced\.venv\Scripts\python.exe
Python 버전: 3.10.6
PyTorch: 2.14.1+cpu
Transformers: 4.57.6
GPU 사용 가능: False


## 2. Qwen 모델 로딩

약 5억 개 파라미터를 가진 지시문 모델을 CPU에서 실행합니다.


In [2]:
# Qwen 모델과 토크나이저 로딩
from transformers import AutoTokenizer, AutoModelForCausalLM

# 사용할 모델 이름
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# 모델 로딩 시작 시간 기록
start_time = time.perf_counter()

# 문장을 토큰 ID로 바꾸는 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name)

# CPU에서 사용할 Qwen 모델 불러오기
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
).to("cpu")

# 학습이 아닌 추론 모드로 설정
model.eval()

# 모델 정보 출력
print("Qwen 모델 로딩 완료")
print("실행 장치:", next(model.parameters()).device)
print(f"로딩 시간: {time.perf_counter() - start_time:.2f}초")
print(f"파라미터 개수: {sum(p.numel() for p in model.parameters()) / 1_000_000:.2f}M")


Qwen 모델 로딩 완료
실행 장치: cpu
로딩 시간: 3.22초
파라미터 개수: 494.03M


## 3. 기본 추론

`apply_chat_template()`는 Qwen의 대화 형식을 적용합니다. 생성 결과에서 입력 토큰을 제외하고 새로 생성된 부분만 출력합니다.


In [7]:
# 사용자 질문 작성
prompt = "로봇의 장애물 회피 방법을 3가지 설명해줘."

# 사용자 역할을 포함한 메시지 만들기
messages = [{"role": "user", "content": prompt}]

# Qwen 대화 형식으로 변환하여 PyTorch 텐서 생성
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

# 기울기 계산 없이 답변 생성
with torch.inference_mode():
    outputs = model.generate(
        inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# 입력 토큰을 제외한 새 토큰 선택
new_tokens = outputs[0][inputs.shape[1]:]

# 새 토큰을 문장으로 변환
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

# 결과 출력
print("질문:", prompt)
print("답변:", answer)
print("생성 토큰 수:", len(new_tokens))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


질문: 로봇의 장애물 회피 방법을 3가지 설명해줘.
답변: 물론이죠! 로봇의 장애물 회피는 여러 가지 방법으로 이루어질 수 있습니다. 여기서는 몇 가지 기본적인 방법들을 소개하겠습니다.

1. **인공지능 (AI) 기반의 장애물 감지 및 예측**: 로봇은 인공지능 기반의 장애물 감지 모듈을 가지고 있어, 이 모듈은 로봇이 어떤 장애물을 발견하거나 방향을 변경할 때까지 계속
생성 토큰 수: 100


In [ ]:
#tokenize=False로 두면 토큰 ID 대신 Qwen 대화 형식의 문자열이 출력. 템플릿이 어떻게 적용되는지 확인하는 데 적합

#prompt = "로봇의 장애물 회피 방법을 3가지 설명해줘."
#messages = [{"role": "user", "content": prompt}]

formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(formatted_text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
로봇의 장애물 회피 방법을 3가지 설명해줘.<|im_end|>
<|im_start|>assistant



## 4. 프롬프트 설계 비교

일반형, 역할형, 조건형 프롬프트를 동일한 생성 설정으로 비교합니다. 

In [9]:
# 서로 다른 유형의 프롬프트 준비
prompts = [
    "로봇 센서를 설명해줘.",
    "너는 로봇공학 교수야. 2학년 학생에게 로봇 센서를 설명해줘.",
    "초음파 센서와 LiDAR를 각각 두 문장으로 비교해줘."
]

# 프롬프트별 결과 저장
prompt_results = []

# 프롬프트를 하나씩 실행
for prompt in prompts:
    # 사용자 메시지 생성
    messages = [{"role": "user", "content": prompt}]

    # 대화 형식을 토큰 ID로 변환
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )

    # 추론 실행 시간 측정
    start = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(
            inputs, max_new_tokens=100, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.perf_counter() - start

    # 새로 생성된 부분만 디코딩
    new_tokens = outputs[0][inputs.shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # 결과 저장
    prompt_results.append({"prompt": prompt, "answer": answer, "seconds": round(elapsed, 2)})
    print("질문:", prompt)
    print("답변:", answer)
    print(f"시간: {elapsed:.2f}초")
    print("-" * 50)


질문: 로봇 센서를 설명해줘.
답변: 물론입니다! 로봇 센서는 로봇의 감지 및 조작 기능을 제공하는 데 사용되는 장치입니다. 이들은 다양한 형태와 기능을 가지고 있으며, 각각의 기능은 다음과 같습니다:

1. 인식: 로봇 센서는 물체의 위치, 크기, 색상, 상태 등을 인식합니다. 예를 들어, 물체가 어떤 방향으로 움직이고 있는지, 어떤 종
시간: 5.22초
--------------------------------------------------
질문: 너는 로봇공학 교수야. 2학년 학생에게 로봇 센서를 설명해줘.
답변: 물론입니다! 로봇 센서는 로봇의 작동을 관리하고, 로봇의 위치와 속도, 그리고 조작을 수행하는 데 사용됩니다. 로봇 센서는 다양한 기술과 분야에서 활용되며, 예를 들어, 자율주행 로봇, 스마트폰, 모바일 애플리케이션 등에 사용될 수 있습니다.

1. 자율주행 로봇:
시간: 5.94초
--------------------------------------------------
질문: 초음파 센서와 LiDAR를 각각 두 문장으로 비교해줘.
답변: 초음파 센서와 LiDAR는 각각 다른 기술과 특징을 가지고 있습니다.

초음파 센서는 물체의 위치 및 속도를 파악하는 데 사용됩니다. 이는 물체의 방향성과 크기에 따라 다양한 형태로 작용하며, 특히 물체의 위치를 정확하게 파악할 수 있는 기능이 있습니다. 예를 들어, 초음파 센서는 물체의
시간: 5.51초
--------------------------------------------------


## 5. 생성 파라미터 실험 준비

- `max_new_tokens`: 새로 생성할 최대 토큰 수
- `do_sample=False`: 가장 확률이 높은 토큰을 선택하는 탐욕적 생성
- `temperature`: 샘플링 확률 분포 조절
- `top_k`: 확률 상위 K개 후보 사용 (`0`은 제한 없음)
- `top_p`: 누적 확률을 기준으로 후보 선택

* `temperature`, `top_k`, `top_p`는 `do_sample=True`에서 비교합니다. 공정한 비교를 위해 같은 입력을 고정합니다.


In [10]:
# 모든 파라미터 실험에 사용할 공통 질문
experiment_prompt = "이동 로봇이 복도에서 장애물을 발견했을 때 어떻게 행동해야 하는지 설명해줘."

# Qwen 대화 형식으로 공통 입력 생성
experiment_inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": experiment_prompt}],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

print("실험 질문:", experiment_prompt)
print("입력 토큰 수:", experiment_inputs.shape[1])


실험 질문: 이동 로봇이 복도에서 장애물을 발견했을 때 어떻게 행동해야 하는지 설명해줘.
입력 토큰 수: 54


## 6. Temperature 비교

In [12]:
# 비교할 Temperature 값
temperatures = [0.3, 0.7, 1.2]

# 온도별 결과를 저장할 리스트
temperature_results = []

# 온도만 바꾸면서 비교
for temp in temperatures:
    # 실험 재현을 위한 난수 시드 설정
    torch.manual_seed(42)

    # 추론 시작 시간 기록
    start = time.perf_counter()

    # 확률 기반 샘플링으로 텍스트 생성
    with torch.inference_mode():
        outputs = model.generate(
            experiment_inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=temp,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

    # 경과 시간과 새로 생성된 토큰 계산
    elapsed = time.perf_counter() - start
    new_tokens = outputs[0][experiment_inputs.shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # 결과 저장 및 출력
    temperature_results.append({
        "experiment": "temperature", "value": temp, "answer": answer,
        "generated_tokens": len(new_tokens), "seconds": round(elapsed, 2)
    })
    print(f"Temperature={temp} | {elapsed:.2f}초")
    print(answer)
    print("-" * 50)


Temperature=0.3 | 5.28초
이동 로봇이 복도에서 장애물을 발견할 때의 행동은 다음과 같습니다:

1. 첫 번째: 장애물 발견 시, 이동 로봇은 먼저 주변 환경에 대한 정보를 수집합니다.

2. 두 번째: 장애물의 위치와 크기 등을 파악하고, 이를 기반으로 적절한 방향
--------------------------------------------------
Temperature=0.7 | 4.98초
이동 로봇이 복도에서 장애물을 발견할 경우, 적절한 조치를 취하기 위해서는 여러 가지 단계가 필요합니다:

1. **알림**: 이동 로봇은 장애물 발견 시 즉시 알림을 내리는 것이 중요합니다. 대신 다른 로봇이나 인공지능을 통해 해당 위치에 대한 정보를 제공
--------------------------------------------------
Temperature=1.2 | 5.20초
먼저, 기존의 로봇이나 바람처럼, 이동하는 시기에 장애물을 만났다거나 보고 있더라도 다음 위치에 다시 출처가 이동할 수 있습니다.

이로부터 일관된 동작을 수행하기 위해서는:

1. **기능설명**:
    - **차세대 디바이스** (AI
--------------------------------------------------


## 7. Top-k와 Top-p 비교

In [7]:
# Top-k와 Top-p의 비교 조건
settings = [
    {"top_k": 10, "top_p": 1.0},
    {"top_k": 50, "top_p": 1.0},
    {"top_k": 0, "top_p": 0.8},
    {"top_k": 0, "top_p": 0.95}
]

# 결과 저장 리스트
sampling_results = []

# 설정별 반복 실험
for setting in settings:
    # 동일한 난수 시드 사용
    torch.manual_seed(42)

    # 추론 시작 시간 기록
    start = time.perf_counter()

    # Top-k 또는 Top-p 설정에 따른 텍스트 생성
    with torch.inference_mode():
        outputs = model.generate(
            experiment_inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_k=setting["top_k"],
            top_p=setting["top_p"],
            pad_token_id=tokenizer.eos_token_id
        )

    # 생성 시간과 응답 추출
    elapsed = time.perf_counter() - start
    new_tokens = outputs[0][experiment_inputs.shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # 실험 결과 저장
    sampling_results.append({
        "experiment": "top_k_top_p",
        "value": f"top_k={setting['top_k']}, top_p={setting['top_p']}",
        "answer": answer, "generated_tokens": len(new_tokens),
        "seconds": round(elapsed, 2)
    })
    print("설정:", setting)
    print("응답:", answer)
    print("-" * 50)


설정: {'top_k': 10, 'top_p': 1.0}
응답: 이동 로봇이 복도에서 장애물을 발견할 경우, 다음과 같은 행동을 취할 수 있습니다:

1. 주의가 떨어진 시점에 즉시 멈춤: 이동 로봇은 항상 주의가 떠있어야 합니다. 장애물이 오는 순간부터 멈출 필요가 있어야 합니다.

2. 최
--------------------------------------------------
설정: {'top_k': 50, 'top_p': 1.0}
응답: 이동 로봇이 복도에서 장애물을 발견하게 되면 적절한 조치를 취하기 위해서는 여러 가지 단계가 필요합니다:

1. **알림 방송**:
   - 이동 로봇은 복도의 높이와 길이 등을 고려하여 최대한 많은 사람들에게 알릴 수 있습니다.

2. **
--------------------------------------------------
설정: {'top_k': 0, 'top_p': 0.8}
응답: 이동 로봇이 복도에서 장애물을 발견할 때의 적절한 행동은 다음과 같습니다:

1. **차단 및 제거**:
   - 이동 로봇이 장애물에 접근하기 전에는 우선 차단되어야 합니다. 이동 로봇이 장애물을 침착하게 두르거나 그와 충돌할 수 있는 부분
--------------------------------------------------
설정: {'top_k': 0, 'top_p': 0.95}
응답: 이동 로봇이 복도에서 장애물을 발견할 경우, 적절한 조치를 취하기 위해서는 여러 가지 단계가 필요합니다:

1. **알림**: 이동 로봇은 장애물 발견 시 즉시 알림을 내리는 것이 중요합니다. 대신 다른 로봇이나 인공지능을 통해 해당 위치에 대한 정보를 제공
--------------------------------------------------


## 8. 생성 토큰 수와 속도 비교

`tokens/s`는 전체 `generate()` 실행 시간을 기준으로 계산한 간단한 지표입니다. 입력 처리 시간도 포함됩니다.


In [8]:
# 최대 생성 토큰 수 비교 조건
token_limits = [30, 60, 100]

# 속도 측정 결과 저장 리스트
speed_results = []

# 토큰 수별 반복 실험
for limit in token_limits:
    # 시작 시간 기록
    start = time.perf_counter()

    # 동일 입력에서 탐욕적 생성 실행
    with torch.inference_mode():
        outputs = model.generate(
            experiment_inputs,
            max_new_tokens=limit,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # 실행 시간 계산
    elapsed = time.perf_counter() - start

    # 실제 생성된 토큰 수 계산
    generated = outputs.shape[1] - experiment_inputs.shape[1]

    # 초당 토큰 수 계산
    speed = generated / elapsed if elapsed > 0 else 0

    # 결과 저장
    speed_results.append({
        "max_new_tokens": limit,
        "generated_tokens": generated,
        "seconds": round(elapsed, 2),
        "tokens_per_second": round(speed, 2)
    })

# 표로 출력
speed_df = pd.DataFrame(speed_results)
print(speed_df.to_string(index=False))


 max_new_tokens  generated_tokens  seconds  tokens_per_second
             30                30     1.64              18.31
             60                60     3.03              19.82
            100               100     4.88              20.51


## 9. CSV 저장 및 결과 확인

In [9]:
# 프롬프트 비교 결과를 CSV 파일로 저장
pd.DataFrame(prompt_results).to_csv(
    "03_Qwen_Prompt_Results.csv", index=False, encoding="utf-8-sig"
)

# 생성 파라미터 비교 결과를 CSV 파일로 저장
pd.DataFrame(temperature_results + sampling_results).to_csv(
    "03_Qwen_Parameter_Results.csv", index=False, encoding="utf-8-sig"
)

# 생성 속도 비교 결과를 CSV 파일로 저장
speed_df.to_csv(
    "03_Qwen_Results.csv", index=False, encoding="utf-8-sig"
)

print("CSV 파일 3개 저장 완료")


CSV 파일 3개 저장 완료


## 10. 실습 정리 및 과제

1. 일반형·역할형·조건형 프롬프트의 출력 차이를 설명하세요.
2. Temperature가 낮을 때와 높을 때 어떤 차이가 있었나요?
3. Top-k와 Top-p의 차이를 설명하세요.
4. 최대 생성 토큰 수가 증가하면 실행 시간은 어떻게 변하나요?

### GitHub 업로드 (Git 저장소가 연결된 경우)

```powershell
git add 03_Qwen_Practice.ipynb 03_Qwen_Prompt_Results.csv 03_Qwen_Parameter_Results.csv 03_Qwen_Results.csv
git commit -m "3회차 Qwen 추론 및 생성 파라미터 실습"
git push
```
